# XGBoost + Optuna — LRG

Para tuning longo, use `python scripts/train/tune_xgb.py --object LRG`.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, RESULTS_DIR as PROJECT_RESULTS, MODELS_DIR, print_info

OBJECT_TYPE = "LRG"
print_info()

paths = paths_for(OBJECT_TYPE)
HDF5_PATH = paths["spectra_h5"].with_name(f"{OBJECT_TYPE}spectra_padded.h5")
print(f"\nObjeto: {OBJECT_TYPE}")
print(f"HDF5:   {HDF5_PATH}")


In [ ]:
import numpy as np, pandas as pd, optuna, xgboost as xgb

from src.data import load_spectral_dataset, normalize_spectra, make_or_load_split, apply_split
from config import SPLITS_DIR
from src.models import tune_xgboost_with_optuna
from src.evaluation import compute_redshift_metrics

SEED = 42
N_TRIALS = 30
RESULTS_DIR = PROJECT_RESULTS / OBJECT_TYPE / "xgboost_optuna"
MODEL_DIR   = MODELS_DIR / OBJECT_TYPE / "xgboost_optuna"
RESULTS_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)


## 2. Dados

In [ ]:
X, y, _ = load_spectral_dataset(HDF5_PATH, seed=SEED)
X = normalize_spectra(X)
# Split canonico estratificado por z (mesmo de todos os modelos).
tr, va, te = make_or_load_split(OBJECT_TYPE, y, SPLITS_DIR)
X_train, X_val, X_test = apply_split(X, tr, va, te)
y_train, y_val, y_test = apply_split(y, tr, va, te)


## 3. Tuning

In [ ]:
storage = f"sqlite:///{MODEL_DIR / f'optuna_xgb_{OBJECT_TYPE.lower()}.db'}"
study = tune_xgboost_with_optuna(X_train, y_train, X_val, y_val,
    n_trials=N_TRIALS, study_name=f"xgboost_{OBJECT_TYPE.lower()}",
    storage=storage, seed=SEED)
print(f"Best RMSE: {study.best_value:.4f}\nBest params: {study.best_params}")


## 4. Treinar final + Salvar

In [ ]:
best = {"objective": "reg:squarederror", "tree_method": "hist", "device": "cpu",
        "random_state": SEED, "n_jobs": -1, **study.best_params}
final = xgb.XGBRegressor(**best)
final.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]), verbose=False)
y_pred = final.predict(X_test)
results = compute_redshift_metrics(y_test, y_pred)

final.save_model(MODEL_DIR / "xgboost_optuna_final.json")
np.savez(RESULTS_DIR / "predictions.npz", y_test=y_test, y_pred=y_pred, delta_z=results["delta_z"])
pd.DataFrame([{"model": "xgboost_optuna", "object": OBJECT_TYPE,
               **{k: v for k, v in results.items() if isinstance(v, (int, float))}}
            ]).to_csv(RESULTS_DIR / "metrics.csv", index=False)
print(f"RMSE: {results['rmse']:.4f}")
